# Step 1 : Import Required Library.

In [1]:
import pandas as pd
import numpy as np

# Step 2 : Load Clean Dataset

In [2]:
df = pd.read_csv("../Dataset/cleaned_data.csv")

# Step 3 : Check Dataset

In [3]:
df.head()

,area_type,availability,location,total_sqft,bath,balcony,price,bhk
0,Super built-up Area,Ready To Move,1st Block BEL Layout,1540.0,3.0,2.0,85.0,3
1,Built-up Area,Ready To Move,1st Block BEL Layout,1800.0,5.0,1.0,250.0,4
2,Super built-up Area,Ready To Move,1st Block HBR Layout,600.0,1.0,0.0,45.0,1
3,Built-up Area,Ready To Move,1st Block HBR Layout,3150.0,4.0,2.0,150.0,4
4,Plot Area,Ready To Move,1st Block HRBR Layout,2400.0,7.0,3.0,355.0,7


In [4]:
df.shape

(9688, 8)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9688 entries, 0 to 9687
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     9688 non-null   str    
 1   availability  9688 non-null   str    
 2   location      9688 non-null   str    
 3   total_sqft    9688 non-null   float64
 4   bath          9688 non-null   float64
 5   balcony       9688 non-null   float64
 6   price         9688 non-null   float64
 7   bhk           9688 non-null   int64  
dtypes: float64(4), int64(1), str(3)
memory usage: 1004.3 KB


# Step 4 : Check Columns

In [6]:
df.columns

Index(['area_type', 'availability', 'location', 'total_sqft', 'bath',
       'balcony', 'price', 'bhk'],
      dtype='str')

# Step 5 : Select Features and Target Variable

## We selected the important columns as input features (X) for our model:
### 1. area_type
### 2. location
### 3. total_sqft
### 4. bath
### 5. balcony
### 6. BHK

## The target variable (y) is:
### 1. price

### The model will use these features to predict the price of a house.

# Step 6 : Drop Unnecessary Columns

In [7]:
df.drop(['availability'], axis=1, inplace=True)

### The `availability` column was removed because it was not important for house price prediction.

## Step 6.1 : Verify Remaining Columns

In [8]:
df.columns

Index(['area_type', 'location', 'total_sqft', 'bath', 'balcony', 'price',
       'bhk'],
      dtype='str')

# Step 7 : Separate X and y

In [9]:
X = df.drop('price', axis=1)
y = df['price']

## Why Feature Engineering?

### Transforming raw data into a format that machine learning algorithms can understand and learn from.

# Step 8 : Check Categorical Columns

In [10]:
df.select_dtypes('object').columns

C:\Users\Ravi Patel\AppData\Local\Temp\ipykernel_22124\258164279.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes('object').columns


Index(['area_type', 'location'], dtype='str')

## Step 8.1 : Handle Location Column

In [11]:
df['location'].nunique()

811

In [12]:
df['location'].value_counts()

location
Whitefield               518
Sarjapur  Road           373
Electronic City          266
Yelahanka                182
Kanakpura Road           176
                        ... 
Udaya Nagar                1
Ullal Road                 1
Vasanthpura                1
Vibhutipura Extension      1
Vyalikaval                 1
Name: count, Length: 811, dtype: int64

### These rare locations increase dimensionality and can hurt model performance.

## Step 8.2 : Group Rare Locations

In [13]:
location_counts = df['location'].value_counts()

location_less_than_10 = location_counts[location_counts <= 10]

df['location'] = df['location'].apply(
    lambda x: 'other' if x in location_less_than_10.index else x
)

### This reduces hundreds of location categories into a manageable number.

##  Step 8.3: Verify Location Count Again

In [14]:
df['location'].nunique()

183

In [15]:
df['location'].value_counts()

location
other                   2110
Whitefield               518
Sarjapur  Road           373
Electronic City          266
Yelahanka                182
                        ... 
Dasanapura                11
Kaikondrahalli            11
Kodigehaali               11
Poorna Pragna Layout      11
Sector 7 HSR Layout       11
Name: count, Length: 183, dtype: int64

### We should see a significant reduction in unique locations.

# Step 9 : One-Hot Encoding

### Machine Learning models cannot understand text values directly. One-Hot Encoding converts categorical values into numerical format.

In [16]:
df = pd.get_dummies(df,columns=['area_type', 'location'],drop_first=True)

## Step 9.1 : Check New Dataset Shape

In [17]:
df.shape

(9688, 190)

 # Step 10 : Separate Features (X) and Target (y)

In [18]:
X = df.drop('price', axis=1)
y = df['price']

- X contains all input features.
- y contains the target variable (price) which we want to predict.

In [19]:
X.shape

(9688, 189)

In [20]:
y.shape

(9688,)

# Step 11 : Train-Test Split

- Training Data (80%) → Model learns from this.
- Testing Data (20%) → Model is evaluated on unseen data.

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)

In [22]:
X_train.shape

(7750, 189)

In [23]:
X_test.shape

(1938, 189)

### Train-Test Split is used to evaluate the model on unseen data and check whether it generalizes well to new examples

# Step 12 : Feature Scaling

### Feature scaling ensures that all numerical features are on a similar scale, which helps some machine learning algorithms perform better.

In [24]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Step 12.1 : Save scaler

In [25]:
import pickle
pickle.dump(scaler, open('../Models/scaler.pkl', 'wb'))

# Step 13 : Save Training and Testing DataSets

In [26]:
X_train.to_csv("../Dataset/X_train.csv", index=False)
X_test.to_csv("../Dataset/X_test.csv", index=False)

y_train.to_csv("../Dataset/y_train.csv", index=False)
y_test.to_csv("../Dataset/y_test.csv", index=False)

# Step 14 : Save Scaler dataSets

In [27]:
pd.DataFrame(X_train_scaled).to_csv("../Dataset/X_train_scaled.csv", index=False)
pd.DataFrame(X_test_scaled).to_csv("../Dataset/X_test_scaled.csv", index=False)